## Messages in LangChain
LangChain uses different message types to represent a chat conversation.
- `HumanMessage` — user's message
- `AIMessage` — model's response
- `SystemMessage` — instructions for the model
- `ToolMessage` — result from a tool call

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model='openai/gpt-oss-20b')

### HumanMessage
The most basic message type. Used to send a user query to the model.

In [ ]:
from langchain_core.messages import HumanMessage

msg = HumanMessage(content='Python kya hai? Short mein batao.')
response = model.invoke([msg])
print(response.content)

### SystemMessage
Used to set the model's role or personality. For example — "you are a Python developer" or "answer in formal tone".

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

msgs = [
    SystemMessage(content='Tum ek experienced Python developer ho. Har answer code example ke saath do.'),
    HumanMessage(content='list comprehension kya hota hai?')
]

resp = model.invoke(msgs)
print(resp.content)

### AIMessage
Represents the model's own response. Used when building multi-turn conversations so the model has context of what it said before.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# multi-turn conversation banana
conversation = [
    SystemMessage(content='Tum ek friendly coding tutor ho.'),
    HumanMessage(content='mujhe for loop samjhao'),
    AIMessage(content='For loop ek iteration tool hai jo kisi sequence (list, string etc) ke har element pe ek ek karke kaam karta hai.'),
    HumanMessage(content='acha iska ek example do fruits wala')
]

resp = model.invoke(conversation)
print(resp.content)

### Tuple format
A shortcut way to write messages using `('role', 'content')` tuples instead of message objects.

In [ ]:
# tuple format — ye bhi kaam karta hai
msgs = [
    ('system', 'Tum ek travel guide ho. India ke baare mein batao.'),
    ('human', 'Rajasthan mein kya dekhne layak hai?')
]

resp = model.invoke(msgs)
print(resp.content)

### Message properties
Each message object has useful properties like `content`, `type`, `id`, and `usage_metadata` (token counts).

In [ ]:
resp = model.invoke([HumanMessage(content='2+2 kitna hota hai?')])

print('Content:', resp.content)
print('Type:', resp.type)
print('ID:', resp.id)
print()

# token usage dekhte hain
if resp.usage_metadata:
    print('Input tokens:', resp.usage_metadata['input_tokens'])
    print('Output tokens:', resp.usage_metadata['output_tokens'])
    print('Total tokens:', resp.usage_metadata['total_tokens'])

### Multi-turn conversation
Building a full conversation where the model remembers previous messages. We need to pass all previous messages each time.

In [ ]:
# full conversation with context
chat_history = [
    SystemMessage(content='Tum ek helpful assistant ho. Short mein jawab do.'),
    HumanMessage(content='mera naam Mohit hai'),
    AIMessage(content='Hello Mohit! Kaise ho?'),
    HumanMessage(content='theek hu. mera naam kya hai?')
]

resp = model.invoke(chat_history)
print(resp.content)
# model ko yaad hona chahiye ki naam Mohit hai

### ChatPromptTemplate
Create reusable prompt templates with variables. Useful when the same prompt format is needed multiple times.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Tum ek {language} expert ho. Sirf {language} mein jawab do.'),
    ('human', '{question}')
])

# chain bana lete hain
chain = prompt | model

resp = chain.invoke({'language': 'Hindi', 'question': 'Machine Learning kya hai?'})
print(resp.content)

In [ ]:
# same template, different language
resp2 = chain.invoke({'language': 'English', 'question': 'What is Deep Learning?'})
print(resp2.content)

### MessagesPlaceholder
Used to dynamically inject chat history into a prompt template. Useful when building chatbots.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Tum ek helpful assistant ho.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

chain = prompt | model

# pehle kuch history bana lete hain
history = [
    HumanMessage(content='mujhe Python sikhna hai'),
    AIMessage(content='Zaroor! Python ek beginner friendly language hai. Kahan se start karna chahoge?')
]

resp = chain.invoke({'history': history, 'input': 'data science ke liye kya seekhun?'})
print(resp.content)